# OCR Pipeline (TrOCR only, no API key)

Fully local handwritten/printed OCR pipeline using TrOCR.

**Flow:**
1. Upload an image -> preprocessing
2. Line segmentation
3. TrOCR extracts the text
4. (Optional) simple offline English spell-check — no internet or API key needed

## 1) Install dependencies

In [ ]:
!pip -q install -U transformers accelerate torch torchvision pillow opencv-python
!pip -q install -U sentencepiece protobuf pyspellchecker


## 2) Imports

In [ ]:
import os
import random

import numpy as np
import cv2
import torch

from PIL import Image, ImageFilter, ImageEnhance

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("Libraries imported successfully.")


Libraries imported successfully.


## 3) Configuration

In [ ]:
# TrOCR model (handwritten English). Use 'microsoft/trocr-base-printed' for printed text.
TROCR_MODEL_NAME = "microsoft/trocr-base-handwritten"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Config loaded. Device:", DEVICE)


Config loaded. Device: cpu


## 4) Load the TrOCR model

In [ ]:
try:
    # Normal path.
    processor = TrOCRProcessor.from_pretrained(TROCR_MODEL_NAME)
except ValueError:
    try:
        # Some transformers versions still fail to auto-build the fast
        # tokenizer even with use_fast=False, so build the tokenizer and
        # image processor separately and assemble them by hand — this
        # skips the AutoProcessor code path entirely.
        from transformers import RobertaTokenizer, ViTImageProcessor

        tokenizer = RobertaTokenizer.from_pretrained(TROCR_MODEL_NAME)
        image_processor = ViTImageProcessor.from_pretrained(TROCR_MODEL_NAME)
        processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
    except Exception as e2:
        raise RuntimeError(
            "Could not load the TrOCR processor/tokenizer. Try: "
            "pip install -U transformers tokenizers sentencepiece protobuf"
        ) from e2

trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL_NAME)
trocr_model.to(DEVICE)

print("TrOCR model loaded on", DEVICE)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

## 5) Image preprocessing

In [ ]:
def preprocess_image(image, rotate_fix=0):
    """Grayscale + contrast/sharpness boost + Otsu threshold, returns a cleaned PIL image."""
    if rotate_fix:
        image = image.rotate(rotate_fix, expand=True)

    gray = image.convert("L")

    gray = ImageEnhance.Contrast(gray).enhance(1.5)
    gray = ImageEnhance.Sharpness(gray).enhance(1.5)

    img_cv = np.array(gray)
    _, thresh = cv2.threshold(img_cv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return Image.fromarray(thresh).convert("RGB")


## 6) Line segmentation

In [ ]:
def segment_lines(image, min_gap=10, min_height=10):
    """Split a page image into line crops using a horizontal projection profile."""
    img = np.array(image.convert("L"))
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    horizontal_sum = np.sum(binary, axis=1)

    lines = []
    in_line = False
    start = 0

    for i, value in enumerate(horizontal_sum):
        if value > min_gap and not in_line:
            start = i
            in_line = True
        elif value <= min_gap and in_line:
            end = i
            if end - start > min_height:
                lines.append(image.crop((0, start, image.width, end)))
            in_line = False

    # If nothing was segmented (single line / no clean gaps), use the whole image
    if not lines:
        lines = [image]

    return lines


## 7) TrOCR text extraction

In [ ]:
def extract_text_from_line(line_image):
    pixel_values = processor(images=line_image, return_tensors="pt").pixel_values.to(DEVICE)
    generated_ids = trocr_model.generate(pixel_values, max_length=128)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


def run_trocr(image_path, rotate_fix=0):
    """Full TrOCR pass over an image: preprocess -> segment -> extract -> join lines."""
    image = Image.open(image_path).convert("RGB")

    clean = preprocess_image(image, rotate_fix=rotate_fix)
    lines = segment_lines(clean)

    line_texts = [extract_text_from_line(line) for line in lines]
    raw_text = "\n".join(line_texts)

    return raw_text, line_texts, lines


## 8) (Optional) simple offline spell-check — English only, no API key

In [ ]:
from spellchecker import SpellChecker

spell = SpellChecker()


def spell_correct(text):
    corrected_lines = []
    changes = []

    for line in text.split("\n"):
        words = line.split(" ")
        new_words = []
        for w in words:
            stripped = w.strip(".,!?;:\"'()")
            if stripped and stripped.isalpha() and stripped.lower() not in spell:
                suggestion = spell.correction(stripped.lower())
                if suggestion and suggestion != stripped.lower():
                    new_words.append(w.replace(stripped, suggestion))
                    changes.append(f"{stripped} -> {suggestion}")
                    continue
            new_words.append(w)
        corrected_lines.append(" ".join(new_words))

    return "\n".join(corrected_lines), changes


## 9) Run it

Change `IMAGE_PATH` to the path of the image you want to OCR.

In [ ]:
IMAGE_PATH = "/content/your_image.jpg"  # <- change this path

raw_text, line_texts, lines = run_trocr(IMAGE_PATH)

print("EXTRACTED TEXT:")
print(raw_text)

# Optional spell-check (English only)
USE_SPELLCHECK = False

if USE_SPELLCHECK:
    corrected_text, changes = spell_correct(raw_text)
    print("\n------------------------")
    print("SPELL-CHECKED TEXT:")
    print(corrected_text)
    print("\nCorrections:")
    for c in changes:
        print("-", c)


---
## 10) (Optional) Fine-tuning TrOCR on your own handwriting dataset

These cells aren't needed for normal OCR to work — only use them if you want to fine-tune the model on your own data (a CSV with `image,text` columns).

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


def add_noise(image):
    img = np.array(image).astype(np.float32)
    noise = np.random.normal(0, 10, img.shape)
    img = np.clip(img + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(img)


def augment_image(image):
    image = image.convert("RGB")

    if random.random() < 0.7:
        image = image.rotate(random.uniform(-5, 5), expand=True, fillcolor="white")
    if random.random() < 0.3:
        image = image.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.2)))
    if random.random() < 0.5:
        image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.5:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.3))
    if random.random() < 0.3:
        image = add_noise(image)

    return image


class OCRDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor, augment=False):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(os.path.join(self.image_dir, row["image"])).convert("RGB")

        if self.augment:
            image = augment_image(image)

        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"], padding="max_length", max_length=128, truncation=True, return_tensors="pt"
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-ocr",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=5,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    save_total_limit=2,
    report_to="none",
    predict_with_generate=True,
)

# train_dataset = OCRDataset(csv_file="/content/train.csv", image_dir="/content/images",
#                             processor=processor, augment=True)
